Loading PDF files from a directory


In [27]:
from langchain_community.document_loaders import PyMuPDFLoader,DirectoryLoader,TextLoader

loader = DirectoryLoader(
    "../data/pdf_files/",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader
)

pdf_files = loader.load()
len(pdf_files[0].page_content)

3275

In [28]:
import re

In [29]:
print(len(pdf_files[0].page_content))
print(repr(pdf_files[0].page_content[:300]))

3275
'PULI ABHINAV   \n \n \n \n \n \n \n \n \n   Mobile: 8328584765 \n          \n                \n \n     Email     LinkedIn       Portfolio  \n  \nPROFESSIONAL SUMMARY  \nA motivated and enthusiastic B.Tech (CSE – Data Science) student with a strong interest in computer science fundamentals \nand data-driven technolog'


In [ ]:
for data in pdf_files:
    text = data.page_content
    text = re.sub(r"[ \t]+"," ",text)
    text = re.sub(r"\s*\n\s*"," ",text)
    text = text.strip()
    data.page_content =  text

In [31]:
print(len(pdf_files[0].page_content))
print(repr(pdf_files[0].page_content[:300]))

2887
'PULI ABHINAV Mobile: 8328584765 Email LinkedIn Portfolio PROFESSIONAL SUMMARY A motivated and enthusiastic B.Tech (CSE – Data Science) student with a strong interest in computer science fundamentals and data-driven technologies. Currently building a solid academic foundation in programming, mathemat'


In [32]:
from langchain_text_splitters import RecursiveCharacterTextSplitter as rcts

In [33]:
splitter = rcts(chunk_size=250, chunk_overlap=50)
final_splits = splitter.split_documents(pdf_files)
print(f"Splitted pdf files into {len(final_splits)} chunks")

Splitted pdf files into 22 chunks


In [62]:
pdf_files[1].page_content

'THAAMMISHETTY ABHINAV B-TECH COMPUTER SCIENCE ENGINEERING STUDENT – DATA SCIENCE STREAM +919880864135 | abhinavthaammishetty@gmail.com LinkedIn: www.linkedin.com/in/abhinavthaammishetty Objective Highly motivated and detail-oriented B-Tech Computer Science Engineering student specializing in Data Science. Proficient in Python, SQL, and machine learning concepts. Hands-on experience in data analysis and visualization, etc. Strong analytical thinker with a deep passion for solving intricate problems in any domain with the help of data science techniques. Seeking an internship or entry-level position to use my skills to be part of some impactful projects in the field of data science. Education Bachelor of Technology (B.Tech) – Computer Science & Engineering (Data Science) Vaagdevi Engineering College | 2024 – 2028 Pre-University College (Class XII) KLE’s Independent PU College Secondary School (Class X) Florence High School TECHNICAL SKILLS • Programming Languages: Python, C, Java • Core

In [35]:
from pathlib import Path
owners = {
    "PULI ABHINAV-CV": "Puli Abhinav",
    "THAAMMISHETTY ABHINAV CV":"Thaammishetty Abhinav"
}

for chunk in final_splits:
    filename = Path(chunk.metadata['source']).stem
    name = owners[filename]
    chunk.metadata['owner'] = name
    chunk.page_content = "cv of " + name + " : " + chunk.page_content

In [36]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_store = InMemoryVectorStore(model)
vector_store.add_documents(documents=final_splits)
print(f"Indexed {len(final_splits)} chunks")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4869.68it/s]


Indexed 22 chunks


In [56]:
from langchain_ollama import ChatOllama

chat = ChatOllama(model="mistral",temperature=0)

In [51]:
def answer_question(question):
    prompt = build_prompt(question=question)
    reply = chat.invoke(prompt)
    return reply.content

In [67]:
def build_prompt(question):
    docs = vec_search(question)
    context = [doc.page_content for doc in docs]
    prompt = f'''  context:{context}
                   question : {question}
                   Answer the question only by using the context provided, 
                   if there is no information related to the question 
                   in context answer by replying I dont have any information regarding your question '''
    return prompt

In [53]:
def vec_search(question):
    docs = vector_store.similarity_search(question,k=3)
    return docs

In [68]:
question="brazilian e-commerce sales analysis project"
answer = answer_question(question=question)
print(answer)

 The context provides information about a project called "Brazilian E-Commerce Sales Analysis" worked on by Puli Abhinav. He analyzed the Olist Brazilian e-commerce dataset end to end, cleaned data with Python, queried it with SQL, and built a 3-page Power BI dashboard on sellers, delivery, logistics, product categories, and payments. However, there is no specific mention of the method used for sales analysis or the results obtained from this project.
